# 02 - Train / validation / test splits

This notebook rebuilds the BBQ table the same way as notebook 01 (`01_bbq_exploration.ipynb`), then adds a `split` column that puts every row in `train`, `val`, or `test`.

For why we split this way and what the final numbers are, see [docs/splits.md](../docs/splits.md).

## Setup

Import the libraries and set the things we use everywhere: where the BBQ files live, the 4 categories we use, the columns that identify a question, and the random seed. `SEED = 600` is fixed, so don't change it (see docs/splits.md for why).

In [ ]:
import pandas as pd
import numpy as np

base = "https://raw.githubusercontent.com/nyu-mll/BBQ/main/data/"
meta_url = "https://raw.githubusercontent.com/nyu-mll/BBQ/main/analysis_scripts/additional_metadata.csv"

categories = ["Gender_identity", "Race_ethnicity", "Age", "SES"]
keys = ["category", "question_index", "example_id"]
SEED = 600

## Load metadata (with duplicates removed)

The metadata file gives us `target_loc`, which tells us which answer option is the stereotyped group. It has some duplicate rows for Gender_identity, so we drop them here. If we didn't, the join below would make extra copies of some questions.

In [ ]:
meta = pd.read_csv(meta_url)
meta_small = meta[keys + ["target_loc"]].drop_duplicates(subset=keys)

## Labeling functions

These are the same functions as in notebook 01. `find_unknown` finds which answer option is "unknown". `build_table` turns each question into 3 rows (one per answer option) and labels each one as correct, biased (the wrong answer that follows the stereotype), or anti-stereotype.

In [ ]:
def find_unknown(info):
    for i in range(3):
        if info[f"ans{i}"][1] == "unknown":
            return i
    return None

def build_table(m):
    rows = []
    for _, q in m.iterrows():
        unk = find_unknown(q["answer_info"])
        target = q["target_loc"]
        non_target = [i for i in range(3) if i not in (unk, target)][0]
        stereo = target if q["question_polarity"] == "neg" else non_target
        for i in range(3):
            is_correct = int(i == q["label"])
            rows.append({
                "row_id": f"{q['category']}_{q['example_id']}_ans{i}",
                "example_id": q["example_id"],
                "template_id": f"{q['category']}_{q['question_index']}",
                "category": q["category"],
                "context_condition": q["context_condition"],
                "question_polarity": q["question_polarity"],
                "context": q["context"],
                "question": q["question"],
                "answer_text": q[f"ans{i}"],
                "answer_idx": i,
                "is_correct": is_correct,
                "is_biased": int(not is_correct and i == stereo),
                "anti_stereotype": int(not is_correct and i != stereo and i != unk),
            })
    return pd.DataFrame(rows)

## Build the full table

For each category, load the BBQ data, join in `target_loc`, drop the questions with no `target_loc`, and build the rows. Then stack all 4 categories into one table. It should have 69240 rows, the same as notebook 01.

In [ ]:
all_tables = []
for cat in categories:
    d = pd.read_json(base + cat + ".jsonl", lines=True)
    m = d.merge(meta_small, on=keys, how="left")
    m = m.dropna(subset=["target_loc"])
    m["target_loc"] = m["target_loc"].astype(int)
    all_tables.append(build_table(m))

table = pd.concat(all_tables, ignore_index=True)
print("total rows:", len(table))   # should be 69240

## Make the splits

We split by template, not by row, because rows from the same template are near-copies (only names are swapped). Inside each category we shuffle the templates with the fixed seed and give 70% to train, 15% to val, and the rest to test. Then every row gets the split of its template. Doing it per category means every category shows up in all 3 splits.

In [ ]:
split_map = {}

for cat in categories:
    templates = sorted(table.loc[table["category"] == cat, "template_id"].unique())
    rng = np.random.default_rng(SEED)
    shuffled = rng.permutation(templates)

    n = len(shuffled)
    n_train = round(0.70 * n)
    n_val = round(0.15 * n)

    for t in shuffled[:n_train]:
        split_map[t] = "train"
    for t in shuffled[n_train:n_train + n_val]:
        split_map[t] = "val"
    for t in shuffled[n_train + n_val:]:
        split_map[t] = "test"

table["split"] = table["template_id"].map(split_map)

## Check the splits

Quick checks that the split worked: how many templates of each category are in each split, how many rows are in each split, and the share of biased rows per split (should be 0.25). The last two lines should both print 0: no template is in more than one split, and every row has a split.

In [ ]:
print(table.groupby(["category", "split"])["template_id"].nunique().unstack(), "\n")
print(table["split"].value_counts(), "\n")
print(table.groupby("split")["is_biased"].mean(), "\n")
print("templates in 2+ splits:", (table.groupby("template_id")["split"].nunique() > 1).sum())
print("rows with no split:", table["split"].isna().sum())